<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Your Places, Week 6 Session 2: Your Places on a Map

### CP101: Introduction to Urban Data Analytics
#### Wednesday, GeoPandas I

Today the fifteen places in `combined_places.csv` stop being numbers in a table and become points
on the Berkeley street map.

**The plan.** The first four sections work through the fifteen places in `combined_places.csv`,
so you can see how each map is made. In section 5 you make the same map for your own favorite places.

**What you submit**

- **In class submission:** **Your own map, from section 5**, saved as an HTML file and uploaded to the
  [Gradescope assignment](https://www.gradescope.com/courses/1333040/assignments/8771148). Submit it before the end of the class for one extra credit towards your class activities, this credit can replace one of your absences!
- **Optional, for 2 bonus points towards your final grade:** Complete the practice at the end, where you combine and clean five My Places
  files from a zip file you have access to in your data/raw folder, and map the result. Once completed, follow the instructions in the notebook to submit to
  [bonus assignment on Gradescope](https://www.gradescope.com/courses/1333040/assignments/8771205) by Monday, October 5, at 12 pm.



<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 1. Point geometry

A `GeoDataFrame` is a `DataFrame` with one special column, `geometry`, that holds shapes rather
than numbers. `points_from_xy` builds that column out of two ordinary ones.

Two things in the `points_from_xy` call below are worth more than they look:

- **`x=` is longitude and `y=` is latitude.** This is the **opposite** of the `(lat, lon)` order
  you wrote in Week 3, and getting it backwards is the single most common mistake in mapping.

In [ ]:
# run this cell to install geopandas and all its dependencies. You only need to do this once per environment.
!pip install "geopandas[all]" -q

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="white")  # seaborn integrates with matplotlib; it only sets the style here: a clean white background for the static maps

In [ ]:
#Load the data and check the first few rows

data_path = "../data/raw/combined_places.csv"
places_df = pd.read_csv(data_path)
places_df.head(3)

`combined_places.csv` holds fifteen places from three lists of five. `lat` and `lon` are two
ordinary numeric columns.

*What happens: a `GeometryArray` of fifteen points. Before you run it, decide whether the first one will read `POINT (37.87 -122.26)` or `POINT (-122.26 37.87)`.*

In [ ]:
# One shapely Point per row. x= is longitude, y= is latitude.
place_geoms = gpd.points_from_xy(
    x=places_df["lon"],     
    y=places_df["lat"]      
    )
place_geoms

Each element is a `shapely` object. Ask for the type of the first one, then look at it.

In [ ]:
type(place_geoms[0])

In [ ]:
place_geoms[0]

Now add the correct shapely geometry to the table, using the geopandas `GeoDataFrame` and declare its CRS (4326).

In [ ]:
places = gpd.GeoDataFrame(
    places_df,
    geometry=place_geoms,
    crs=4326,          # 4326 = plain longitude and latitude, in degrees
)
print("CRS:", places.crs)
places[["name", "category", "geometry"]]

The geometry prints as `POINT (-122.2592 37.8724)`: longitude first. Latitude and
longitude are how people say a location; `x` and `y` are how a drawing surface thinks about it,
and on a map `x` runs east, which is longitude. Both orders are correct, neither is more
natural, and nothing in the data records which one you meant.

These numbers are degrees: fine for drawing a map, not for measuring. For distances in metres, areas,
or any other measurement, convert the layer to a metre-based CRS with `.to_crs()` first.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 2. Add a Street Network Layer

Start with the base layer on its own.

`.parquet` is the main data format for geospatial data that we use in this course. Parquet files usually load faster and take up less space than CSV, GeoJson and Shapefile (for sure!), while preserving data types such as numbers and dates. 



In [ ]:
roads = gpd.read_parquet("data/raw/roads.parquet")
print("road segments:", len(roads))
print("bounds (left, bottom, right, top):", roads.total_bounds)

roads.plot(
    color="lightgray", # the color of the lines
    linewidth=0.7      # the width of the lines
    )
plt.show()

This map covers the area around campus, about 3 km across. 

Run the cell below to plot 15 places. Can you tell where they are without streets in the background?

In [ ]:
places.plot()
plt.show()

Add streets to show where the places are. Using `ax=ax` puts both plots on the same map: grey streets first, then colored points.

Try swapping the two `.plot()` calls for roads and places. Does the map change?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

roads.plot(ax=ax, color="lightgray", linewidth=0.7)             # base layer, drawn first
places.plot(ax=ax, color="#e4572e", markersize=110,             # your places, drawn on top
            edgecolor="white", linewidth=1.3)

ax.set_title("My places on the Berkeley street map")
ax.set_axis_off()
plt.show()

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: colour by a column

**Color places by category.** Replace `places.plot(...)` in section 2 with:

   ```python
   places.plot("category", ax=ax, legend=True, markersize=110)
   ```

   What do the colors show now?

<details>
<summary>Check your answer</summary>

Each color shows a different category, labeled in the legend. Before, all points were the same color.

</details>

In [ ]:
# Your code here

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 3. Static and Interactive Maps 

`.plot()`, as we have seen, will output a static map.

`.explore()` creates an interactive map you can pan and zoom with a basemap that you can change with `tiles` argument. The default is OSM basemap. 

In [ ]:
places.explore(
    column="category",  # the column to color the points by
    tooltip=["name", "category", "accessibility"], #the columns to show when you hover over a point
    tiles="Stadia.AlidadeSmooth" # the tile basemap layer to use
    )

### Style it

`.explore()` takes the same `column=` and `legend=` as `.plot()`, plus a few options that only
make sense on a web map.

| Argument | What it does |
|---|---|
| `categorical=True` | treats `category` as labels, one colour each, not a number scale |
| `cmap="Set1"` | the palette those colours come from |
| `tiles="Stadia.AlidadeSmoothDark"` | a dark basemap, so the colours stand out |
| `marker_type="circle_marker"` | a circle that stays the same size on screen as you zoom |
| `marker_kwds={"radius": 5}` | that circle's size, in screen pixels |
| `style_kwds={...}` | outline width and fill opacity |



In [ ]:
places.explore(
    column="category", # the column to color the points by
    categorical=True,  # whether the column is categorical 
    cmap="Set1",       # the color map to use
    legend=True,       # whether to show a legend
    tiles="Stadia.AlidadeSmoothDark", # the tile basemap layer to use
    marker_type="circle_marker",      # the type of marker to use
    marker_kwds={"radius": 5},        # the radius of the circle markers
    style_kwds={"weight": 1, "fillOpacity": 0.8},  # the style of the markers, here we set the weight of the outline and the fill opacity
    tooltip=["name", "category"],  # the columns to show when you hover over a point
)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 4. Add Parks to your Map

Another layer you have access to is `parks.parquet`. 
Add it to show parks in green.
Note that in the code below, Parks layer is the first layer to be mapped and other layers will be drawn on top of it.

Uncomment the lines that have # behind them and run again to see what changes

In [ ]:
parks = gpd.read_parquet("data/raw/parks.parquet")
print("parks in the extract:", len(parks))

fig, ax = plt.subplots(figsize=(9, 9)) # create a figure and axis to plot our map on
parks.plot(                   # Parks layer 
    ax=ax,                    # use the canvas we just created
    color="#cfe8cf")        
roads.plot(                   # Roads layer
    ax=ax,                    # draw on the same canvas, over the parks
    color="lightgray",        
    linewidth=0.7             
    ) 
places.plot(
    ax=ax,                    # draw on the same canvas, over the roads and parks
    color="#e4572e",        # hex color for the points
    markersize=90, 
    linewidth=1.2
    )
ax.set_title("Parks, streets, places: three layers, drawn in that order")
ax.set_axis_off()             
# minx, miny, maxx, maxy = places.total_bounds    # get the bounds of the places layer 
# pad = 0.006
# ax.set_xlim(minx - pad, maxx + pad)
# ax.set_ylim(miny - pad, maxy + pad)
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 5. Map Your own Places

Now make the same map for **your own** list of favorite places. You can add more places to your csv if you like!

1. **Remember where your csv is!.** You created your file in lab 2, so relative to where your current notebook is, that file is 2 directory up and then inside lab week 2: 

 `../../../lab/w02/nameofyourfile.csv` replace nameofyourfile with the correct file name. 

2. **Fill in the `...`** in the code below. Use the patterns you saw in section 1 or section 3 or change the style and coloring.
3. **Check the map.** Pan and zoom until you can see that each point sits where the place really is.


In [ ]:
my_path = ...     # change the path to the correct location of your csv file. 

my_df = pd.read_csv(my_path)
my_places = gpd.GeoDataFrame(
    my_df,
    geometry=gpd.points_from_xy(x=..., y=...),
    crs=...,
)
print(len(my_places), "places")

my_map = my_places.explore(
    column=...,
    categorical=True,
    cmap="Set1",
    legend=True,
    tiles="Stadia.AlidadeSmoothDark",
    marker_type="circle_marker",
    marker_kwds={"radius": 6},
    style_kwds={"weight": 1, "fillOpacity": 0.8},
    tooltip=["name", "category"],
)
my_map.save("my_places_map_w6_s2.html")   # the file you submit
my_map

<hr style="border: 1px solid #fdb515;" />

### ✅ Submit your map (can replace one absence)

1. In the file browser, right-click `my_places_map_w6_s2.html` and choose **Download**.
2. Open the downloaded file in your browser and check it works: it should pan, zoom, show all
   five of your places, and show a name when you hover over a point.
3. Upload the file to the [Gradescope assignment](https://www.gradescope.com/courses/1333040/assignments/8771148). This submission can replace one of
   your absences.

That is the class activity done. The practice below is optional.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Practice (2 bonus points): combining five students' files

`data/raw/student_files.zip` contains five  My Places CSV files from a previous CP101 class. Each has 20–30 places.

The files were renamed `student_1_myplaces.csv` through `student_5_myplaces.csv` to remove names and IDs from the filenames. Their contents, including any possible mistakes, are unchanged.

These files use `disability_access` instead of `accessibility` and have no `rating` column. Neither difference prevents us from mapping the places.

Your job:

1. **Unzip the files.** Run this code:

   ```python
   import zipfile
   with zipfile.ZipFile("data/raw/student_files.zip") as z:
       z.extractall("data/raw/")
   ```

   The CSVs will be in `data/raw/student_files/`.

2. **Read and combine the five files** with `pd.concat`. Add a `source` column to track each row’s original file.
3. **Check and clean the data.** Look for differences in column names, categories, and coordinates.
4. **Make an interactive map** with `.explore()`. Color the points by category, use a Stadia basemap, and save it with `.save("students_map_w6_s2.html")`.

<details>
<summary>Hints</summary>

- **Find the files:** Use `sorted(glob.glob("data/raw/student_files/*_myplaces.csv"))` after `import glob`.
- **A file won't open:** If you get `UnicodeDecodeError`, read that file with `encoding="latin-1"`.
- **`note` and `notes` columns:** Rename `note` to `notes` before combining.
- **Repeated categories:** Use `.str.strip().str.lower()` to remove extra spaces and make capitalization consistent.
- **Points far from Berkeley:** Check rows where `combined["lon"] > 0`. If their addresses are in Berkeley, restore the missing minus sign.
- **A Berkeley address appears in San Francisco:** Its coordinates belong to another branch. For this exercise, keep latitude `37.7`–`38.0` and longitude `-122.4`–`-122.1`. Print how many rows you remove.
- **Different spellings of the same name:** Use `.str.strip().str.lower()` before counting how many files mention each place.

</details>

**Expected result:** 116 rows and 7 categories. Yogurt Park appears in 4 of the 5 files.

In [ ]:
# Your code here

<hr style="border: 1px solid #fdb515;" />

### ✅ Package and submit the bonus (2 points)

Due Monday, October 5, at 12 pm:

1. Run your cells so the combined table and the map show below them.
2. **Save the notebook** (`Cmd + S` or `Ctrl + S`). The zip copies the file as it was last saved.
3. Run the cell below. It packs this notebook and `students_map_w6_s2.html` into
   `w6_s2_bonus_submission.zip`.
4. In the file browser, right-click `w6_s2_bonus_submission.zip` and choose **Download**.
5. Upload the zip to the [bonus assignment on Gradescope](https://www.gradescope.com/courses/1333040/assignments/8771205).

In [ ]:
import zipfile
from datetime import datetime
from pathlib import Path

NOTEBOOK = Path("my_places_w6_s2.ipynb")
STUDENTS_MAP = Path("students_map_w6_s2.html")
SUBMISSION = Path("w6_s2_bonus_submission.zip")

assert NOTEBOOK.exists(), f"Can't find {NOTEBOOK}. Open the notebook from its own folder and run this again."
assert STUDENTS_MAP.exists(), f"Can't find {STUDENTS_MAP}. Run your map cell, including the .save() line, first."

saved = datetime.fromtimestamp(NOTEBOOK.stat().st_mtime)
print("Notebook last saved at", saved.strftime("%H:%M:%S"), "- if that is not just now, save and run this again.")

with zipfile.ZipFile(SUBMISSION, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(NOTEBOOK)
    z.write(STUDENTS_MAP)

with zipfile.ZipFile(SUBMISSION) as z:
    print("Created", SUBMISSION, "containing", z.namelist())

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary

| Step | Tool |
|---|---|
| Build point geometry | `gpd.points_from_xy(x=lon, y=lat)`, named arguments |
| Declare what the numbers are | `crs=4326` |
| Measure distance or area | `.to_crs(...)` to a metre-based CRS first |
| Read a saved layer | `gpd.read_parquet(path)` |
| Stack layers | one `ax`, passed to every `.plot()`, in draw order |
| Zoom to your data | `minx, miny, maxx, maxy = gdf.total_bounds` |
| Check it by hand | `.explore()` |
| Save an interactive map | `m = gdf.explore(...)`, then `m.save("name.html")` |
